# Stage 2 Notebook 51 - Exp2VV anchor + VFL + full 70K + det rescue (lambda_det=2.0)

**Fix the joint-training conflict NB48 exposed.** NB48 (anchor + VFL + full data) hit project geometry record (matched_iou=0.544) but val_det stalled at 3.16 and val_map50 = 0.0 -- detection completely failed to fit. The lane gradient at 23x data variety dominated the shared backbone and starved the det branch. The grad_cos diagnostic from earlier experiments already showed lane and det gradients fight in late epochs; at full data scale the conflict becomes catastrophic.

Diagnosis: Kendall uncertainty weighting (`use_uncertainty: true`) learns task weights from the loss magnitudes -- but at full data scale lane loss decreases faster than det loss, so the learned weight shifts AWAY from det, accelerating the imbalance. Fix: switch to fixed weighting and explicitly boost det.

Config diff vs NB48 (Exp2SS):
- `lambda_det: 1.0 -> 2.0`
- `lambda_lane: 1.0 -> 0.7`
- `use_uncertainty: true -> false`
- All else identical to NB48 including VFL recipe + full dataset + 6 epochs.

Reference: this is the standard 'fixed multi-task weighting' fallback when learnable weights run away (Sener & Koltun 2018 'Multi-Task Learning as Multi-Objective Optimization' discusses the failure mode).

### Run mode

1. `DEBUG_MODE = True` smoke.
2. `DEBUG_MODE = False` for full-dataset 6-epoch run.
3. Wall-clock ~ 60-80 min.
4. Independent of all prior NBs.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint_smoke.log
OK exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.0238 det_loss=3.5207 grad_cos=0.0932 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5012975335121155, 'gate/lane_mean': 0.501221239566803, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full6'
    EPOCHS = 6
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint_full6 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint_full6.tar --epochs 6 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint_full6.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp46_rmt_gca_anchor_vfl_full_data_det_rescue_joint_full6_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp46_rmt_gca_anchor_vfl_full_data

0

## What to watch in Exp2VV training

Reference NB48 (anchor + VFL + full data, uncertainty=true, lambda_det=1.0): matched_iou=0.544 (record), oracle_f1=0.467, decoded_f1=0.050, **val_det=3.16** (broken), **val_map50=0.0**.

Pass criteria at epoch 6:
- **`val_det <= 2.2`** and **`val/det/map50 >= 0.005`** -- detection actually trains at full data scale.
- **`val/matched_line_iou >= 0.50`** -- preserve geometry from NB48 (small acceptable regression from 0.544 because lane gradient is dialed back from lambda=1.0 to 0.7).
- **`val/lane/decoded_oracle_f1 >= 0.40`**.
- **`val/lane/decoded_f1 >= 0.04`** -- maintain NB48 level.
- `train/grad_cosine_epoch_mean` should stay positive across most epochs -- confirms the conflict is fixed by the static rebalancing.

Failure signals:
- val_det still >= 3.0: lambda_det=2.0 not enough; raise to 3.0 in a follow-up.
- matched_iou drops below 0.40: lambda_lane=0.7 too aggressive; raise to 0.9.
- BOTH val_det and matched_iou regress: the joint conflict is structural and we need Exp2WW (PCGrad gradient surgery).